# Step 2: Cohort Definition — Type 2 Diabetes RWE Study

**Study objective:** Define an adult Type 2 Diabetes Mellitus (T2DM) cohort for an observational comparative effectiveness study comparing diabetes medication treatment patterns and outcomes.

**Research question:** Among adult patients with Type 2 Diabetes, compare treatment patterns and outcomes between different diabetes medications.

**Data source:** Synthea synthetic EHR CSV exports (`data/synthea_csv`)

**Design:** Active-comparator cohort with index date anchored to first post-diagnosis diabetes pharmacotherapy — standard pharmacoepidemiologic practice for new-user treatment comparisons.

## Study design overview

| Element | Definition |
| --- | --- |
| **Target population** | Adults with confirmed Type 2 Diabetes Mellitus |
| **Index date** | First diabetes medication exposure after T2DM diagnosis |
| **Treatment groups** | Metformin users vs Other diabetes medication users |
| **Inclusion** | Age ≥ 18, T2DM diagnosis, ≥1 encounter, post-diagnosis medication |
| **Diabetes identification** | Primary condition description: `Diabetes mellitus type 2 (disorder)` |

Complication codes (e.g., diabetic neuropathy) are excluded from diagnosis identification to ensure the index diagnosis reflects incident T2DM rather than downstream sequelae.

In [ ]:
from pathlib import Path
import sys

import pandas as pd

NOTEBOOK_DIR = Path.cwd()
PROJECT_ROOT = NOTEBOOK_DIR.parent if NOTEBOOK_DIR.name == "notebooks" else NOTEBOOK_DIR
SRC_DIR = PROJECT_ROOT / "src"
DATA_DIR = PROJECT_ROOT / "data" / "synthea_csv"
REPORT_DIR = PROJECT_ROOT / "reports"

if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

from cohort_definition import (
    load_study_data,
    identify_diabetes_cohort,
    define_treatment_exposure,
    build_study_population,
    run_cohort_definition,
)
from baseline_characteristics import run_baseline_characteristics

## Inclusion and exclusion criteria

**Include patients who:**
1. Are **≥ 18 years** of age at index date
2. Have a diagnosis of **Type 2 Diabetes Mellitus** (conditions.csv, description-based)
3. Have **at least one healthcare encounter** (encounters.csv)
4. Have **diabetes medication exposure after T2DM diagnosis** (medications.csv)

**Treatment assignment (at index date):**
- **Group A — Metformin users:** first post-diagnosis diabetes drug description contains metformin
- **Group B — Other diabetes medication users:** first post-diagnosis diabetes drug is non-metformin (e.g., insulin, GLP-1 agonist)

In [ ]:
# Load source tables
tables = load_study_data(DATA_DIR)

print("Source data loaded:")
for name, df in tables.items():
    print(f"  {name:15s} {len(df):>10,} rows")

In [ ]:
# Run cohort definition pipeline and persist outputs
result = run_cohort_definition(DATA_DIR, REPORT_DIR)

cohort_flow = result.cohort_flow.to_dataframe()
diabetes_cohort = result.diabetes_cohort
treatment_exposure = result.treatment_exposure
study_population = result.study_population

## Cohort flow — patients at each filtering step

CONSORT-style attrition table documenting how inclusion criteria reduce the source population to the final analytic cohort.

In [ ]:
display(cohort_flow.style.format({"n": "{:,}"}))

# Waterfall chart of cohort attrition
flow_plot = cohort_flow.copy()
flow_plot["step_label"] = flow_plot["step"].str.replace(r"^\d+_", "", regex=True)

import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(flow_plot["step_label"], flow_plot["n"], marker="o", linewidth=2, color="#1d4ed8")
ax.set_ylabel("Number of patients")
ax.set_title("Cohort Size at Each Filtering Step")
plt.xticks(rotation=30, ha="right")
plt.tight_layout()
plt.show()

## Diabetes cohort and treatment exposure

In [ ]:
print(f"T2DM patients identified: {len(diabetes_cohort)}")
display(diabetes_cohort.head())

print(f"\nTreatment exposure records: {len(treatment_exposure)}")
display(
    treatment_exposure.groupby("treatment_group")
    .agg(n=("patient_id", "count"), index_medication=("index_medication", "first"))
)

## Final study population

The analytic cohort (`reports/study_population.csv`) includes demographics and treatment assignment at the index date (first diabetes medication after T2DM diagnosis).

In [ ]:
display(study_population.head(10))

summary = (
    study_population.groupby("treatment_group")
    .agg(
        n=("patient_id", "count"),
        mean_age=("age", "mean"),
        female_pct=("gender", lambda s: (s == "F").mean() * 100),
    )
    .round(1)
)
print("\nSummary by treatment group:")
display(summary)

## Baseline characteristics

Descriptive statistics for the final cohort: age distribution, gender, treatment group sizes, and prevalent comorbidities at or before index date.

In [ ]:
baseline = run_baseline_characteristics(DATA_DIR, REPORT_DIR)
display(baseline)

## Next steps in the RWE pipeline

With cohort definition complete, subsequent steps typically include:

1. **Baseline covariate assembly** — labs (HbA1c, BMI), utilization, socioeconomic factors
2. **Propensity score estimation** — balance measured confounders across treatment groups
3. **Outcome ascertainment** — clinical endpoints and follow-up time
4. **Treatment effect estimation** — IPTW, Cox models, sensitivity analyses

Outputs from this step are archived under `reports/`:
- `study_population.csv` — final analytic cohort
- `cohort_flow.csv` — inclusion/exclusion attrition
- `diabetes_cohort.csv`, `treatment_exposure.csv` — intermediate artifacts
- `baseline_characteristics.csv` — descriptive statistics